# 04 — Run the Optimize wizard and export a candidate

**Foundry feature:** the **Optimize wizard** itself — eval/optimization model selection, running
the search, and exporting the winning candidate. **Mode: Portal, then CODE.**

> **No live Foundry access in this environment.** The cells below that would normally happen in the
> Foundry portal (creating the agent, running the wizard, exporting a candidate) are written as
> numbered manual instructions, not executable code — Foundry's Optimize wizard is portal-only in
> the current preview. To keep the rest of the series runnable end-to-end without portal access,
> notebook 04 writes a **stand-in candidate file** to the same path a real export would use. Swap it
> for your own exported candidate the moment you have portal access, and every notebook downstream
> keeps working unchanged.

In [ ]:
import json, subprocess, sys
from pathlib import Path

# Repo layout: this notebook lives in notebooks/, the pack lives in ../prompt-agent-optimizer-baselines
PACK_ROOT = Path("..").resolve() / "prompt-agent-optimizer-baselines"
AGENT_ID = "01-travel-approval-strict"          # <- the one case study every notebook in this series uses
AGENT_DIR = PACK_ROOT / AGENT_ID

assert AGENT_DIR.exists(), f"Can't find {AGENT_DIR} -- run this notebook from a checkout of the repo."

def run(cmd, cwd=PACK_ROOT):
    """Run a pack CLI tool and print its output, the way you would from a terminal."""
    print("$", " ".join(cmd))
    result = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print(result.stderr)
    return result

print(f"Pack root : {PACK_ROOT}")
print(f"Case study: {AGENT_ID}")

## Portal — model selection

The wizard asks you to pick two models:

- **Optimization model** — proposes the rewritten instructions.
- **Eval model** — judges each candidate's responses against your dataset's `ground_truth`.

Pick the eval model from a **vendor family disjoint from the optimization model** — the same rule
this pack's own `judge_config` follows, to avoid a same-family self-preference bias where a model
rates its own family's output more favorably (Zheng et al. 2023; Panickssery et al. 2024).

In [ ]:
jc = json.loads((AGENT_DIR / "expected" / "expectations.json").read_text())["judge_config"]
print("This pack's own convention for this agent (mirror it in the wizard's model pickers):")
print(f"  primary judge / eval model : {jc['primary_judge_model']}")
print(f"  cross judge (same-family)  : {jc['cross_judge_model']}  (bias check only, never the deciding verdict)")
print(f"  {jc['note']}")

## Portal — run and export

1. Select the optimization model and eval model as above.
2. Start the run and wait for it to complete — the wizard proposes and scores several candidate
   instruction sets.
3. Open the winning candidate, and **export its instructions** as plain text or Markdown.
4. Save the exported file as `candidates/foundry_run1.md` inside the case study's folder.

## CODE — since this environment has no portal access

Everything downstream in this series (validation, judging, holdout scoring, comparison) needs a real
candidate file at that path. Without portal access here, this cell writes a **stand-in candidate** —
the baseline instructions with the one documented flaw (the hedged sign-off line) actually fixed, so
the rest of the series has something realistic to validate. **The moment you have a real exported
candidate, overwrite this file with it** — nothing downstream needs to change.

In [ ]:
candidates_dir = AGENT_DIR / "candidates"
candidates_dir.mkdir(exist_ok=True)
candidate_path = candidates_dir / "foundry_run1.md"

baseline_text = (AGENT_DIR / "instructions.md").read_text()
stand_in_text = baseline_text.replace(
    "Sometimes it's nice to add a friendly sign-off to make the traveller feel supported.",
    "Close with a one-line summary of the decision and the next action the traveller should take.",
)
assert stand_in_text != baseline_text, "expected replacement text not found in instructions.md"
candidate_path.write_text(stand_in_text)
print(f"Wrote stand-in candidate to {candidate_path}")
print("Replace this file with your real Foundry-exported candidate when you have portal access.")

## CODE — record a run manifest

Foundry's optimizer is a hosted, versioned, non-deterministic service — a later "Run 1 vs Run 2"
comparison is meaningless without knowing whether anything upstream changed between runs. Copy
`_tools/run_manifest_template.json` per run and fill in what the portal showed you. This cell does
that for the stand-in run above; fill in the real values (model version IDs, timestamp, wizard
config) once you're recording an actual portal run.

In [ ]:
import datetime

runs_dir = PACK_ROOT / "runs" / AGENT_ID
runs_dir.mkdir(parents=True, exist_ok=True)

manifest = json.loads((PACK_ROOT / "_tools" / "run_manifest_template.json").read_text())
manifest.update({
    "run_id": "run1-notebook-demo",
    "agent_id": AGENT_ID,
    "run_label": "run1",
    "timestamp_utc": datetime.datetime.utcnow().isoformat() + "Z",
    "optimization_model": {"name": "REPLACE_ME e.g. gpt-5.1", "version_id": "REPLACE_ME"},
    "eval_model": {"name": jc["primary_judge_model"], "version_id": "REPLACE_ME"},
})
optimize_row_count = sum(1 for l in (AGENT_DIR / "dataset" / "optimize.jsonl").read_text().splitlines() if l.strip())
manifest["dataset"]["row_count"] = optimize_row_count
manifest["outputs"]["candidate_instructions_path"] = str(candidate_path.relative_to(PACK_ROOT))
manifest["notes"] = "Stand-in candidate generated by notebook 04 -- replace with a real portal export."

manifest_path = runs_dir / "run1.manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2))
print(f"Wrote {manifest_path}")

## Next

Continue to **`05_validate_candidate_contract.ipynb`** to score this candidate against the case
study's contract.